# Vehicle Classification Using Custom CNN

# Import Important Library


In [1]:
# Standard Library for data
import numpy as np
import pandas as pd 
import os
import shutil
import zipfile
from pathlib import Path 
import cv2
from PIL import Image
import random

# keras
from tensorflow.keras.layers import Input, Lambda, Dense, Flatten
from tensorflow.keras.models import Model
from tensorflow.keras.applications.vgg16 import VGG16
from tensorflow.keras.preprocessing import image
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img
from tensorflow.keras.models import Sequential
import numpy as np

import warnings
warnings.filterwarnings("ignore")

# Load The Datasets

In [2]:
# ekstrak zip
def extract_zip(zip_path,extract_to="."):
    print(f"Mengekstrak {zip_path}")
    with zipfile.ZipFile(zip_path,"r") as zip_ref:
        zip_ref.extractall(extract_to)
    print("Extraksi Selesai\n")

# split the dataset
def split_dataset(source_dir,output_dir,train_ratio=0.7,val_ratio=0.15,test_ration=0.15,seed=42):
    random.seed(seed)
    
    source_path = Path(source_dir)
    output_path = Path(output_dir)
    
    for split in ["train","val","dir"]:
        for class_dir in source_path.iterdir():
            if class_dir.is_dir():
                (output_path/split/class_dir.name).mkdir(parents=True,exist_ok=True)
    
    for class_dir in source_path.iterdir():
        if not class_dir.is_dir():
            continue
        
        class_name = class_dir.name
        images = [f for f in class_dir.iterdir() if f.suffix.lower() in [".jpg",".jpeg",".png"]]
        random.shuffle(images)
        
        total = len(images)
        train_end = int(total*train_ratio)
        val_end = train_end + int(total*val_ratio)
        
        splits = {
            "train" : images[:train_end],
            "val" : images[train_end:val_end],
            "test" : images[val_end:]
        }
        
        for split_name,split_images in splits.items():
            for img_path in split_images:
                dest = output_path/split_name/class_name/img_path.name
                shutil.copy2(img_path,dest)
                
        print(f"[{class_name}] Total: {total} → Train: {len(splits['train'])}, Val: {len(splits['val'])}, Test: {len(splits['test'])}")

In [3]:
# specified the destinec file
ZIPFILE = "archive.zip"
SOURCE = "Vehicles"
DEST = "ImageClasiDataset"

# apply the extraction
extract_zip(ZIPFILE,extract_to=".")
split_dataset(SOURCE,DEST)

Mengekstrak archive.zip


KeyboardInterrupt: 

In [ ]:
def load_dataset(folder: Path, split: str, img_size=(224, 224)):
    X = []
    y = []

    split_path = folder / split 

    for class_dir in sorted(split_path.iterdir()):
        if not class_dir.is_dir():
            continue

        class_name = class_dir.name 

        for img_path in class_dir.iterdir():
            if img_path.suffix.lower() not in ['.jpg', '.jpeg', '.png']:
                continue

            img = Image.open(img_path).convert('RGB')   
            img = img.resize(img_size)                  
            X.append(np.array(img))                     
            y.append(class_name)                 

    return np.array(X), np.array(y)

In [ ]:
# Apply the function
X_Train,Y_Train = load_dataset(DEST,"train")
X_Val,Y_Val = load_dataset(DEST,"val")
X_Test,Y_Test = load_dataset(DEST,"test")